In [ ]:
from minio import Minio
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length 
from pyspark.sql.types import NumericType, IntegerType, LongType, FloatType, DoubleType, DecimalType, DateType, TimestampType
from pyspark.sql.functions import window
from pyspark.sql import Window
import pyspark.sql.functions as F
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os

In [ ]:
JAR_PATH_1 = os.path.abspath("./jars/hadoop-aws-3.4.0.jar")
JAR_PATH_2 = os.path.abspath("./jars/aws-sdk-s3-2.29.52.jar")

JARS_LIST = f"{JAR_PATH_1},{JAR_PATH_2}"

In [ ]:
spark = (
    SparkSession.builder.appName("analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.jars.repositories", "https://repo1.maven.org/maven2/")
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

# Customer Related Analysis 

Adding Date Column

In [ ]:
dataframes["customers"] = dataframes["customers"].withColumn(
    "account_created_date",
    F.to_date("account_created_at")
)

Time Grain function

In [ ]:
def add_time_grain(df, date_col="account_created_date", grain="day"):
    if grain == "day":
        return df.withColumn("grain_date", F.col(date_col))
    elif grain == "week":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_week", F.weekofyear(date_col))
    elif grain == "month":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_month", F.month(date_col))
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

active customers over time

In [ ]:

def active_customers_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.filter(F.col("is_active") == True)
          .groupBy(*group_cols)
          .agg(F.countDistinct("customer_id").alias("active_customers"))
          .orderBy(*group_cols)
    )

daily_active   = active_customers_over_time(dataframes["customers"], "day")
weekly_active  = active_customers_over_time(dataframes["customers"], "week")
monthly_active = active_customers_over_time(dataframes["customers"], "month")

account_status over time 

In [ ]:
def status_distribution_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.groupBy(*(group_cols + [F.col("account_status")]))
          .agg(F.countDistinct("customer_id").alias("customer_count"))
          .orderBy(*group_cols, "account_status")
    )

daily_status   = status_distribution_over_time(dataframes["customers"], "day")
monthly_status = status_distribution_over_time(dataframes["customers"], "month")

New customers per day/week/month

In [ ]:
def new_customers(df, grain="day"):
    df_g = add_time_grain(df, grain=grain)

    if grain == "day":
        group_cols = ["grain_date"]
        order_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
        order_cols = ["grain_year", "grain_week"]
    else:   # month
        group_cols = ["grain_year", "grain_month"]
        order_cols = ["grain_year", "grain_month"]

    new_df = (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .orderBy(*order_cols)
    )
    return new_df

daily_new   = new_customers(dataframes["customers"], "day")
weekly_new  = new_customers(dataframes["customers"], "week")
monthly_new = new_customers(dataframes["customers"], "month")

Cumulative customer growth curve

In [ ]:
def cumulative_customers(df, grain="day"):
    new_df = new_customers(df, grain)

    # Define window by time order
    if grain == "day":
        window = Window.orderBy("grain_date") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    elif grain == "week":
        window = Window.orderBy("grain_year", "grain_week") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    else:   # month
        window = Window.orderBy("grain_year", "grain_month") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    cum_df = new_df.withColumn(
        "cumulative_customers",
        F.sum("new_customers").over(window)
    )
    return cum_df

daily_growth   = cumulative_customers(dataframes["customers"], "day")
weekly_growth  = cumulative_customers(dataframes["customers"], "week")
monthly_growth = cumulative_customers(dataframes["customers"], "month")

Total new customers by geography + time

In [ ]:
geo_acquisition = (
    dataframes["customers"]
    .groupBy("country", "state_province", "city")
    .agg(F.countDistinct("customer_id").alias("new_customers"))
)

def geo_acquisition_over_time(df, grain="day"):
    df_g = add_time_grain(df, grain=grain)

    if grain == "day":
        group_cols = ["grain_date", "country", "state_province", "city"]
        order_cols = ["grain_date", "country", "state_province", "city"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
    else:  # month
        group_cols = ["grain_year", "grain_month", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_month", "country", "state_province", "city"]

    return (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .orderBy(*order_cols)
    )

daily_geo   = geo_acquisition_over_time(dataframes["customers"], "day")
monthly_geo = geo_acquisition_over_time(dataframes["customers"], "month")